### Set up

In [1]:
# imports

from explore_import import  *
import ionbot_preprocess as io
import data_preprocess as dt
import tpp_preprocess as tpp
import hpp_checker as hpp

import pyteomics.auxiliary as aux
from pathlib import Path
import os, re, subprocess
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# base directories

root="/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery"
data_dir=f"{root}/oui-discovery-vv-data/raw/msfragger_pride_reanalysis/fragpipe23_ms42/"
processed_dir=f"{root}/oui-discovery-vv-data/processed/msfragger_pride_reanalysis/fragpipe23_ms42/"
r_path="Rscript"
gwalk_work_dir=root
gwalk_script="Run_group_walk_tppfragpipe.R"

In [3]:
# replicate insides of data directory to preprocessed directory

#get insides of data directory
data_paths=dt.list_files(data_dir)

#replicate
for parent in data_paths.keys():
    new_parent=parent.replace(data_dir, processed_dir)
    os.makedirs(new_parent, exist_ok=True)

/
trembl/
    PXD014258/
        ESC_HF_SampleHela/
            psm.tsv
            peptide.tsv
            filelist_proteinprophet.txt
            filter.log
            interact.mod.pep.xml
            fragger.params
            log_2025-05-27_11-36-28.txt
            shepherd.config
            fragpipe.workflow
            ion.tsv
            combined.prot.xml
            sdrf.tsv
            interact.pep.xml
            fragpipe-files.fp-manifest
            ptm-shepherd-output/
                global.profile.tsv
                global.modsummary.tsv
        ESC_HF_Sample_BT474/
            interact.pep.xml
            log_2025-05-27_10-17-10.txt
            ion.tsv
            peptide.tsv
            filelist_proteinprophet.txt
            sdrf.tsv
            fragger.params
            psm.tsv
            combined.prot.xml
            interact.mod.pep.xml
            shepherd.config
            fragpipe.workflow
            filter.log
            fragpipe-files.fp-manifest
     

In [4]:
# modification variables

expected_mods={57.0215,15.9949, 42.0106}
expected_mods_name=['Iodoacetamide derivative/Addition of Glycine/Addition of G','Oxidation or Hydroxylation','Acetylation']
unmodified_threshold=0.01

### Load and process PeptideProphet files

In [5]:
def classify_leadprot(x):
    x=x.replace("decoy_","")
    if 'CONTAMINANT' in x.upper():
        return 'Contam'
    elif x.startswith('II_') or x.startswith('IP_'):
        return 'NonCanon'
        # Ensembl is canonical
    else:
        return 'Canon'

def is_peptide_canonical(x):
    '''x is the list of protein classes'''
    if np.array([_=='Contam' for _ in x]).any():
        return 'Contam'
    if np.array([_=='Canon' for _ in x]).any():
        return 'Canonical'
    return 'NonCanonical'

def custom_subgroup_filter(data_,key):
    filtered_subgroups = []
    for c,df in data_.groupby("FDRGroup").__iter__():
        tmp = aux.target_decoy.qvalues(df, key=key, reverse=True, is_decoy=df.database=='D',
                                      formula=1, full_output=True, q_label='custom_q')
        filtered_subgroups.append(tmp)

    return pd.concat(filtered_subgroups, ignore_index=True)

In [6]:
def annotate_massshift(value,ptm_df,col):
    #print(value)
    a=ptm_df[(ptm_df['peak_lower'] <= value) & (ptm_df['peak_upper'] >= value)]
    if len(a[col])==0 or a[col].isna().all(): 
        return None
    return "||".join(set([str(x).split("shift ")[-1] if "Unidentified" in str(x) or "Unannotated" in str(x) else str(x) for x in a[col].values.tolist()]))

In [7]:
def col_isna(df,col):
    return 100-round((df[col].isna().value_counts()[False]/len(df))*100,4)

In [8]:
data_paths

{'/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/raw/msfragger_pride_reanalysis/fragpipe23_ms42/trembl/PXD014258/ESC_HF_SampleHela': ['psm.tsv',
  'peptide.tsv',
  'filelist_proteinprophet.txt',
  'filter.log',
  'interact.mod.pep.xml',
  'fragger.params',
  'log_2025-05-27_11-36-28.txt',
  'shepherd.config',
  'fragpipe.workflow',
  'ion.tsv',
  'combined.prot.xml',
  'sdrf.tsv',
  'interact.pep.xml',
  'fragpipe-files.fp-manifest'],
 '/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/raw/msfragger_pride_reanalysis/fragpipe23_ms42/trembl/PXD014258/ESC_HF_SampleHela/ptm-shepherd-output': ['global.profile.tsv',
  'global.modsummary.tsv'],
 '/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/raw/msfragger_pride_reanalysis/fragpipe23_ms42/trembl/PXD014258/ESC_HF_Sample_BT474': ['interact.pep.xml',
  'log_2025-05-27_10-17-10.txt',
  'ion.tsv',
  'peptide.tsv',


In [9]:
qval_score='peptideprophet_probability'

for parent, files in data_paths.items():
    new_parent=parent.replace(data_dir,processed_dir)
    database, dataset, spectrum_file = parent.split("/")[-3:]
    spectrum_file = spectrum_file.replace("_","-") if dataset=="PXD014258" else spectrum_file
    for file in files:
        if file.endswith(".mod.pep.xml"):
            # read PeptideProphet output
            pepxml_path=os.path.join(parent, file)
            data=pepxml.DataFrame(pepxml_path)
            print(f"{dataset}|{database}|{spectrum_file} N of discoveries: {len(data)}")

            #pin dataset and search_database
            data["dataset"] = dataset
            data["search_database"] = database
            data["spectrum_file"] = spectrum_file

            #pin T/D database
            data["database"]=data["protein"].apply(tpp.get_database_tpp)
            td_counts = round((data["database"].value_counts()/len(data))*100,4)
            print(f"{dataset}|{database}|{file} targets and decoys and NA: {td_counts['T']} and {td_counts['D']} and {col_isna(data,'database')}")

            #calculate global q-value on peptideprophet_probability
            print(f"{dataset}|{database}|{file} {qval_score} range and NA: {min(data[qval_score])} - {max(data[qval_score])} and {col_isna(data,qval_score)}")
            data = aux.target_decoy.qvalues(data,
                                            key=qval_score,
                                            reverse=True,
                                            is_decoy=(data.database == 'D'),
                                            q_label='global_q',
                                            formula=1,
                                            full_output=True)
            print(f"{dataset}|{database}|{file} 'global_q' range and NA: {min(data['global_q'])} - {max(data['global_q'])} and {col_isna(data,'global_q')}")

            #pin helpfull labels
            data["peptide_class"]=data["protein"].apply(tpp.classify_peptide_tpp)
            
            #pin subgroups
            data['protein_classes'] = data.protein.apply(lambda x: np.unique([classify_leadprot(_) for _ in x]))
            data['isCanonical'] = data.protein_classes.apply(is_peptide_canonical)
            
            #add modifications info
            ptmshepherd_file=os.path.join(parent, "ptm-shepherd-output", "global.profile.tsv")
            ptmshepherd_data = pd.read_csv(ptmshepherd_file, sep="\t")
            #correct columns read
            cols=ptmshepherd_data.columns.tolist()
            ptmshepherd_data=ptmshepherd_data.reset_index()
            ptmshepherd_data.columns=cols+["na"]
            #mass shift annotation
            data['massdiff']=data['massdiff'].astype(float)

            massdiffs={float(_):{"mapped_mass_1":[],"mapped_mass_2":[]} for _ in data.massdiff.unique()}
            for md in massdiffs.keys():
                sub=ptmshepherd_data[(ptmshepherd_data.peak_lower<=md)&(md<=ptmshepherd_data.peak_upper)]
                mapped_mass_1, mapped_mass_2 = sub.mapped_mass_1.tolist() , sub.mapped_mass_2.tolist()
                massdiffs[md]["mapped_mass_1"]= None if len(mapped_mass_1)==0 else "||".join([str(_) for _ in mapped_mass_1])
                massdiffs[md]["mapped_mass_2"]= None if len(mapped_mass_2)==0 else "||".join([str(_) for _ in mapped_mass_2])
            data=data.merge(pd.DataFrame(massdiffs).T.reset_index(), left_on="massdiff", right_on="index", how="left")
            data.loc[((data.mapped_mass_1.str.contains(expected_mods_name[0])) | (
            data.mapped_mass_1.str.contains(expected_mods_name[1])) | 
            (data.mapped_mass_1.str.contains(expected_mods_name[2]))) & (data.mapped_mass_2=='nan'),"isModified"]="Expected"
            data.loc[((data.mapped_mass_1=="nan")|(data.mapped_mass_1.isna()))&(abs(data.massdiff)<=unmodified_threshold),"isModified"]="Unmodified"
            data.loc[(data.isModified.isna()),"isModified"]="Unexpected"
            data.drop(["index_x","index_y"],axis=1,inplace=True)
            
            data['isTarget'] = data.database.apply(lambda x: x=='T')
            #data['FDRGroup'] = data.isCanonical + '_' + data.isModified
            data['FDRGroup'] = data.isCanonical
            fdrgroup_counts = round((data["FDRGroup"].value_counts()/len(data))*100,4)
            counts_str = ", ".join(
                f"{key} {fdrgroup_counts[key]}%"
                for key in list(fdrgroup_counts.keys())
            )
            na_count = col_isna(data, 'FDRGroup')
            print(f"{dataset}|{database}|{file} {counts_str}, NA {na_count}")

            # calculate group-wise q-value
            data = custom_subgroup_filter(data, qval_score)
            print(f"{dataset}|{database}|{file} 'custom_q' range and NA: {min(data['custom_q'])} - {max(data['custom_q'])} and {col_isna(data,'custom_q')}")
            
            #save for Group-walk
            gwalk_input = os.path.join(new_parent, f"{spectrum_file}.csv")
            data.to_csv(gwalk_input,index=False)
            
            #run Group-walk
            dataset_dir = new_parent
            working_dir = gwalk_work_dir
            file_name = os.path.basename(gwalk_input)
            print(f"Run Group-walk on {gwalk_input}")
            command = (
                f"module load r && {r_path} {gwalk_script} {dataset_dir} {working_dir} {file_name}"
            )
            
            _ = subprocess.run(command, shell=True, check=True)
            
            qwalk_output = os.path.join(new_parent,"groupwalk_output_"+file_name)

            #check group-walk
            data2=pd.read_csv(qwalk_output)
            print(f"{dataset}|{database}|{file} 'group_q_prob' range and NA: {min(data2['group_q_prob'])} - {max(data2['group_q_prob'])} and {col_isna(data2,'group_q_prob')}")
            del data2


#            break
#    break

PXD014258|trembl|ESC-HF-SampleHela N of discoveries: 30559
PXD014258|trembl|interact.mod.pep.xml targets and decoys and NA: 93.3538 and 6.6462 and 0.0
PXD014258|trembl|interact.mod.pep.xml peptideprophet_probability range and NA: 0.05 - 1.0 and 0.0
PXD014258|trembl|interact.mod.pep.xml 'global_q' range and NA: 0.0 - 0.0711932136848009 and 0.0
PXD014258|trembl|interact.mod.pep.xml Canonical 82.506%, Contam 17.494%, NA 0.0
PXD014258|trembl|interact.mod.pep.xml 'custom_q' range and NA: 0.0 - 0.08653307476836888 and 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/msfragger_pride_reanalysis/fragpipe23_ms42/trembl/PXD014258/ESC_HF_SampleHela/ESC-HF-SampleHela.csv
PXD014258|trembl|interact.mod.pep.xml 'group_q_prob' range and NA: 0.0003667033370003 - 0.071228266965788 and 0.0
PXD014258|trembl|ESC-HF-Sample-BT474 N of discoveries: 37423
PXD014258|trembl|interact.mod.pep.xml targets and decoys and NA: 94.1026 and 5.897

### PSM -> peptide

In [10]:
processed_paths=dt.list_files(processed_dir)

/
canon/
    PXD014258/
        ESC_HF_Sample_MCF/
            ESC-HF-Sample-MCF.csv
            groupwalk_output_ESC-HF-Sample-MCF.csv
            ptm-shepherd-output/
        ESC_HF_Sample_BT474/
            ESC-HF-Sample-BT474.csv
            groupwalk_output_ESC-HF-Sample-BT474.csv
            ptm-shepherd-output/
                groupwalk_output_130327_o2_04_hu_P2_2hr_pep.csv
                130327_o2_05_hu_C3_2hr_pep.csv
                groupwalk_output_AM10_pep.csv
                groupwalk_output_AM7_pep.csv
                groupwalk_output_AM14_pep.csv
                groupwalk_output_AM18_pep.csv
                AM21_pep.csv
                ESC_HF_Sample_MCF_pep.csv
                130327_o2_06_hu_P3_2hr_pep.csv
                AM13_pep.csv
                AM8_pep.csv
                AM17_pep.csv
                ESC_HF_SampleHela_pep.csv
                groupwalk_output_130327_o2_06_hu_P3_2hr_pep.csv
                AM12_pep.csv
                AM9_pep.csv
                AM1

In [41]:
qval_score='peptideprophet_probability'

for parent, files in processed_paths.items():
    database, dataset, spectrum_file = parent.split("/")[11:14]
    for file in files:
        # omit group-walk output PSM-level and newly added peptide-level and group-walk output peptide-level
        if (not file.startswith("groupwalk_output_")) and (not file.endswith("_pep.csv")) :

            data = pd.read_csv(os.path.join(parent, file))
            print(f"PSM -> peptide : {file}")
            data.drop(['global_q','custom_q'],inplace=True,axis=1)

            # select high-scoring PSMs
            data_peptide = data.sort_values(qval_score, ascending=False).drop_duplicates("peptide", keep="first")
            print(f"Number of PSMs and peptides: {len(data)} , {len(data_peptide)}")
            del data

            #pin T/D database
            td_counts = round((data_peptide["database"].value_counts()/len(data_peptide))*100,4)
            print(f"{dataset}|{database}|{file} targets and decoys and NA: {td_counts['T']} and {td_counts['D']} and {col_isna(data_peptide,'database')}")

            #calculate global q-value
            print(f"{dataset}|{database}|{file} {qval_score} range and NA: {min(data_peptide[qval_score])} - {max(data_peptide[qval_score])} and {col_isna(data_peptide,qval_score)}")
            data_peptide = aux.target_decoy.qvalues(data_peptide,
                                            key=qval_score,
                                            reverse=True,
                                            is_decoy=(data_peptide.database == 'D'),
                                            q_label='global_q',
                                            formula=1,
                                            full_output=True)
            print(f"{dataset}|{database}|{file} 'global_q' range and NA: {min(data_peptide['global_q'])} - {max(data_peptide['global_q'])} and {col_isna(data_peptide,'global_q')}")

            # calculate group-wise q-value
            fdrgroup_counts = round((data_peptide["FDRGroup"].value_counts()/len(data_peptide))*100,4)
            counts_str = ", ".join(
                f"{key} {fdrgroup_counts[key]}%"
                for key in list(fdrgroup_counts.keys())
            )
            na_count = col_isna(data_peptide, 'FDRGroup')
            print(f"{dataset}|{database}|{file} {counts_str}, NA {na_count}")
            data_peptide = custom_subgroup_filter(data_peptide, qval_score)
            print(f"{dataset}|{database}|{file} 'custom_q' range and NA: {min(data_peptide['custom_q'])} - {max(data_peptide['custom_q'])} and {col_isna(data_peptide,'custom_q')}")

            #save for Group-walk
            gwalk_input = os.path.join(parent, f"{spectrum_file}_pep.csv")
            data_peptide.to_csv(gwalk_input,index=False)

            #run Group-walk
            dataset_dir = parent
            working_dir = gwalk_work_dir
            file_name = os.path.basename(gwalk_input)
            print(f"Run Group-walk on {gwalk_input}")
            command = (
                f"module load r && {r_path} {gwalk_script} {dataset_dir} {working_dir} {file_name}"
            )
            
            _ = subprocess.run(command, shell=True, check=True)
            
            qwalk_output = os.path.join(parent,'groupwalk_output_'+file_name)

            #check group-walk
            data_peptide2=pd.read_csv(qwalk_output)
            print(f"{dataset}|{database}|{'groupwalk_output_'+file_name} 'group_q_prob' range and NA: {min(data_peptide2['group_q_prob'])} - {max(data_peptide2['group_q_prob'])} and {col_isna(data_peptide2,'group_q_prob')}")
            #del data_peptide2
            
#            break
#        break

PSM -> peptide : ESC-HF-Sample-MCF.csv
Number of PSMs and peptides: 42352 , 20097
PXD014258|canon|ESC-HF-Sample-MCF.csv targets and decoys and NA: 94.2031 and 5.7969 and 0.0
PXD014258|canon|ESC-HF-Sample-MCF.csv peptideprophet_probability range and NA: 0.05 - 1.0 and 0.0
PXD014258|canon|ESC-HF-Sample-MCF.csv 'global_q' range and NA: 0.0 - 0.061536023663638285 and 0.0
PXD014258|canon|ESC-HF-Sample-MCF.csv Canonical 95.6362%, Contam 4.3638%, NA 0.0
PXD014258|canon|ESC-HF-Sample-MCF.csv 'custom_q' range and NA: 0.0 - 0.06358253555420286 and 0.0
Run Group-walk on /project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/processed/msfragger_pride_reanalysis/fragpipe23_ms42/canon/PXD014258/ESC_HF_Sample_MCF/ESC_HF_Sample_MCF_pep.csv
PXD014258|canon|groupwalk_output_ESC_HF_Sample_MCF_pep.csv 'group_q_prob' range and NA: 0.0001168770453482 - 0.0615888442848088 and 0.0
PSM -> peptide : ESC-HF-Sample-BT474.csv
Number of PSMs and peptides: 40292 , 20906
PXD01425

### protein inference

In [36]:
def classify_leadprot(x):
    x=x.replace("decoy_","")
    if 'CONTAMINANT' in x.upper():
        return 'Contam'
    elif x.startswith('II_') or x.startswith('IP_'):
        return 'NonCanon'
        # Ensembl is canonical
    else:
        return 'Canon'

def is_peptide_canonical(x):
    '''x is the list of protein classes'''
    if np.array([_=='Contam' for _ in x]).any():
        return 'Contam'
    if np.array([_=='Canon' for _ in x]).any():
        return 'Canonical'
    return 'NonCanonical'
    
def get_max_psm_score(x,psms,psm_score):
    protein_group = x.split(',')
    tmp3 = psms[psms.protein.isin(protein_group)]
    return np.max(tmp3[psm_score])

def group_is_decoy(x):
    protein_group = x.split(',')
    tmp = [_.startswith('decoy') for _ in protein_group]
    return set(tmp)

def classify_protein_group(x):
    protein_group = x.split(',')
    tmp = [classify_leadprot(_) for _ in protein_group]
    return is_peptide_canonical(tmp)

In [12]:
pipeline="fragpipe"
psm_score="peptideprophet_probability"
#savefile_path = f'{pipeline}-identified-protein-groups-all-datasets.csv.gz'
PXDs = [
    'PXD002057',
    'PXD005833',
    'PXD014258',
]

SEARCHES = [
    'canon',
    'trembl',
    'openprot',
]
DATASETS = pd.MultiIndex.from_product([PXDs,SEARCHES])
DATASETS

MultiIndex([('PXD002057',    'canon'),
            ('PXD002057',   'trembl'),
            ('PXD002057', 'openprot'),
            ('PXD005833',    'canon'),
            ('PXD005833',   'trembl'),
            ('PXD005833', 'openprot'),
            ('PXD014258',    'canon'),
            ('PXD014258',   'trembl'),
            ('PXD014258', 'openprot')],
           )

In [42]:
#  Select peptide-level files
folders = {search:{dataset_name:[] for dataset_name in PXDs} for search in SEARCHES}
for dataset_name in PXDs:
    for search in SEARCHES:   
        path = os.path.join(processed_dir, search, dataset_name)
        flds = os.listdir(path)
        for fld in flds:
            fld = os.path.join(path, fld)
            for subfld in glob.glob(f"{fld}/*.csv"):
                if "_pep.csv" in subfld and "groupwalk_output_" in subfld and not "_Occam" in subfld:
                    folders[search][dataset_name].append(subfld)

In [ ]:
qval_name='global_q'
output_files = []
for dataset_name in PXDs:
    for search in SEARCHES:  
        j = len(folders[search][dataset_name])
        for i,sample_fld in enumerate(folders[search][dataset_name]):          
            
            sample=sample_fld.name.replace(".csv","")
            sample_path=os.path.join(processed_dir, dataset_name, f"{dataset_name}-{search}")
            out_path=os.path.join(sample_path, f"{sample}_protein_groups_Occam.csv")
            
            # Read and preprocess the dataset
            psms = pd.read_csv(os.path.join(sample_fld.path))
            print(f'({1+i}/{j})',sample_fld, psms.shape)

            #DO PEPTIDE BY SAMPLE FILTER
            #psms = psms[psms['global_q']<=.01].copy(deep=True)

            # try without contaminant peptides
            psms.protein=psms.protein.apply(lambda x: ast.literal_eval(x))
            psms['protein_classes'] = psms.protein.apply(lambda x: np.unique([classify_leadprot(_) for _ in x]))
            psms['isCanonical'] = psms.protein_classes.apply(is_peptide_canonical)
            
            #WHAT BEST PUT INFO IN LINE HERE? - MODIFICATIONS AND CHARGE ARE NOT NEEDED?
            with open(os.path.join(sample_path, f"{sample}-protein-inference-input.pout"), 'w') as POUT:
                for _,row in psms[(psms.isCanonical!='Contam')&(psms.protein.apply(len)<1000)].iterrows():
                    row2 =  [str(row["index"]), row.spectrum, f"{row[qval_name]:.2e}", 
                             row.modifications, f"{'n'}.{row.peptide}"] #no charge info
                    row2 += [f'>xx|{_}|placeholder' if '|' not in _ else _ for _ in row.protein]
                    row2 =  '\t'.join(row2) + '\n'
                    POUT.write(row2)
            
            pout_file = os.path.join(sample_path, f"{sample}-protein-inference-input.pout")
            groups_file = os.path.join(sample_path, f"{sample}-groups_Occam.tsv")
            subgroups_file = os.path.join(sample_path, f"{sample}-subgroups_Occam.tsv")
            _ = subprocess.run(['pout2prot', '--occam', pout_file, groups_file, subgroups_file])
            print('Run pout2prot:',_.returncode==0)  
        
            # Make "summary" table to get max psm score
            psms = psms[['protein',psm_score]].copy(deep=True)
            psms = psms.explode('protein')
            psms.sort_values(psm_score, ascending=False, inplace=True)
            psms.drop_duplicates('protein', inplace=True)
        
            # calculate protein groups scores
            protein_groups = pd.read_csv(groups_file, sep='\t', usecols=['protein accessions'])
            protein_groups.columns = ['proteins']
            protein_groups[psm_score] = protein_groups.proteins.apply(lambda x: get_max_psm_score(x,psms,psm_score))
        
            # label decoys
            protein_groups['isDecoy_'] = protein_groups.proteins.apply(group_is_decoy)
            protein_groups['database'] = protein_groups.isDecoy_.apply(lambda x: 'D' if list(x)[-1] else 'T')
    
            protein_groups = protein_groups[['proteins','database',psm_score]].copy(deep=True)
            protein_groups['experiment'] = sample
            protein_groups['exp_number'] =  f'{dataset_name} exp {i+1}' #'-'.join(sample_fld.name.split('-')[:-1])
            protein_groups['PXD'] = dataset_name
            protein_groups['searchDB'] = search
        
            # recalculate q-values
            protein_groups = aux.target_decoy.qvalues(protein_groups, key=psm_score, reverse=True, 
                                                      is_decoy=protein_groups.database=='D',
                                                      q_label='global_q', formula=1, full_output=True)
        
            protein_groups['isCanonical'] = protein_groups.proteins.apply(classify_protein_group)
            protein_groups['isModified'] = 'NA'
            protein_groups = custom_subgroup_filter(protein_groups,psm_score)
            out_path = os.path.join(sample_path, f"{sample}_protein_groups_Occam.csv")
            protein_groups.to_csv(out_path, index=False)

            # RUN groupWalk
            
            output_files.append(out_path)